In [1]:
#| default_exp hyperparameter_tuning

## Hyperparameter Tuning

Peshbeen provides two powerful tools for hyperparameter tuning: `hyperopt_tune` and `optuna_tune`. These functions allow you to optimize the hyperparameters of your forecasting models using the `hyperopt` and `optuna` libraries, respectively. Both functions support cross-validation and can be used with any of the forecasting models available in peshbeen.

### hyperopt_tune example for univariate forecasting using machine learning models

In [2]:
from peshbeen.datasets import load_wales_admissions
from peshbeen.metrics import RMSE
from lightgbm import LGBMRegressor
from peshbeen.models import ml_forecaster
from peshbeen.model_selection import hyperopt_tune
load_wales_admissions["day_of_week"] = load_wales_admissions.index.dayofweek
load_wales_admissions["month"] = load_wales_admissions.index.month
# split the data into train and test sets
train = load_wales_admissions[:-30]
test = load_wales_admissions[-30:]
cat_variables = ["day_of_week", "month"]
# import linear regression from sklearn
ml_model = ml_forecaster(model=LGBMRegressor(verbose=-1),
              target_col='admissions', lags = 30,
              cat_variables=cat_variables)
ml_model.fit(train)

# Define the hyperparameter search space for LightGBM
from hyperopt import hp
from hyperopt.pyll import scope
lgb_param_space={'learning_rate': hp.uniform('learning_rate', 0.001, 0.6),
            'num_leaves': scope.int(hp.quniform('num_leaves', 10, 200, 1)),
           'max_depth':scope.int(hp.quniform('max_depth', 2, 18, 1)),
            'bagging_fraction': hp.uniform('bagging_fraction', 0.5, 1),
            'feature_fraction': hp.uniform('feature_fraction', 0.5, 1),
            'lambda_l2' : hp.uniform('lambda_l2', 0,10),
           'lambda_l1' : hp.uniform('lambda_l1', 0, 10),
           'top_rate' : hp.quniform('top_rate', 0.05, 0.4, 0.0001),
            'other_rate' : hp.quniform('other_rate', 0.05, 0.3, 0.0001),
           'num_iterations': scope.int(hp.quniform("num_iterations", 30, 700, 1)),
           'lags': hp.choice("lags", [
                                 [1,2,3,4,5],
                                 [1,4,7],
                                 [1,2,3,4,5,6,7],
                                 [1,2,3,4,5,6,7,14],
                                 [1,2,3,4,5,6,7,14,21],
                                 [1,2,3],
                             ]),
                "seed":0,
                "box_cox": hp.uniform("box_cox", 0.0, 4),
                "box_cox_biasadj": hp.choice("box_cox_biasadj", [True, False])}

# Run hyperparameter tuning using hyperopt
best_params, best_lags, other_ = hyperopt_tune(model=ml_model, df=train, cv_split=5, step_size=10,
                                        test_size=1, eval_metric=RMSE, eval_num=10,
                                        param_space=lgb_param_space)

print("Best params:", best_params)
print("Best lags:", best_lags)
print("Other info:", other_)

100%|██████████| 10/10 [00:02<00:00,  3.56trial/s, best loss: 95.3477737903806] 
Best params: {'bagging_fraction': 0.6196760108221376, 'feature_fraction': 0.5219568011976503, 'lambda_l1': 1.4582216679681026, 'lambda_l2': 4.554185240699607, 'learning_rate': 0.062102936788035815, 'max_depth': 5, 'num_iterations': 371, 'num_leaves': 186, 'other_rate': 0.2029, 'seed': 0, 'top_rate': 0.2136}
Best lags: [1, 2, 3, 4, 5, 6, 7]
Other info: {'box_cox': 2.8879012378042526, 'box_cox_biasadj': False}


In [3]:
# now we can run our model with the best paramaters, best lags and other info such as box_cox and box_cox_biasadj
best_box_cox = other_["box_cox"]
best_box_cox_biasadj = other_["box_cox_biasadj"]

ml_model = ml_forecaster(model=LGBMRegressor(**best_params, verbose=-1),
              target_col='admissions', lags = list(best_lags), box_cox=best_box_cox, box_cox_biasadj=best_box_cox_biasadj,
              cat_variables=cat_variables)
ml_model.fit(train)
ml_forecasts = ml_model.forecast(H=30, exog=test[cat_variables])
ml_forecasts

array([8868.55556422, 8780.08688274, 8766.85176813, 8939.23345343,
       8876.63667107, 8891.8215311 , 8864.98645826, 8854.29595123,
       8798.99256387, 8778.57339062, 8921.41246967, 8873.85675703,
       8857.808037  , 8839.78404018, 8837.5561451 , 8783.82870339,
       8726.5607891 , 8837.46668516, 8828.05680202, 8825.10663569,
       8822.29332886, 8814.58807221, 8720.64880457, 8697.45093293,
       8761.03906486, 8782.81492049, 8780.54973623, 8806.13333794,
       8787.76499827, 8663.79159922])

### optuna_tune example for univariate forecasting

In [4]:
from peshbeen.datasets import load_wales_admissions
from peshbeen.metrics import MAE, RMSE
from lightgbm import LGBMRegressor
from peshbeen.models import ml_forecaster
from peshbeen.model_selection import optuna_tune
load_wales_admissions["day_of_week"] = load_wales_admissions.index.dayofweek
load_wales_admissions["month"] = load_wales_admissions.index.month
# split the data into train and test sets
train = load_wales_admissions[:-30]
test = load_wales_admissions[-30:]
cat_variables = ["day_of_week", "month"]
# import linear regression from sklearn
ml_model = ml_forecaster(model=LGBMRegressor(verbose=-1),
              target_col='admissions', lags = 30,
              cat_variables=cat_variables)
ml_model.fit(train)
# ml_model.data_prep(train)
forecasts = ml_model.forecast(H=30, exog=test[cat_variables])
lgb_param_space = {
    "learning_rate":     lambda t: t.suggest_float("learning_rate", 0.001, 0.6),
    "num_leaves":        lambda t: t.suggest_int("num_leaves", 10, 200),
    "max_depth":         lambda t: t.suggest_int("max_depth", 2, 18),
    "bagging_fraction":  lambda t: t.suggest_float("bagging_fraction", 0.5, 1.0),
    "feature_fraction":  lambda t: t.suggest_float("feature_fraction", 0.5, 1.0),
    "lambda_l2":         lambda t: t.suggest_float("lambda_l2", 0.0, 10.0),
    "lambda_l1":         lambda t: t.suggest_float("lambda_l1", 0.0, 10.0),
    "top_rate":          lambda t: t.suggest_float("top_rate", 0.05, 0.4),
    "other_rate":        lambda t: t.suggest_float("other_rate", 0.05, 0.3),
    "num_iterations":    lambda t: t.suggest_int("num_iterations", 30, 700),
    "lags":              lambda t: t.suggest_categorical(
                             "lags", [
                                 [1,2,3,4,5],
                                 [1,4,7],
                                 [1,2,3,4,5,6,7],
                                 [1,2,3,4,5,6,7,14],
                                 [1,2,3,4,5,6,7,14,21],
                                 [1,2,3],
                             ]),
                             
    "seed":              lambda t: 0,   # fixed, not sampled
}

best_params, best_lags, other_ = optuna_tune(
    model=ml_model,
    df=train,
    cv_split=10,
    step_size=10,
    test_size=30,
    eval_metric=RMSE,
    eval_num=10,
    param_space=lgb_param_space, verbose=False
)
print("Best params:", best_params)
print("Best lags:", best_lags)

Best params: {'learning_rate': 0.0904885130326401, 'num_leaves': 120, 'max_depth': 10, 'bagging_fraction': 0.5881193969575922, 'feature_fraction': 0.6889832506204177, 'lambda_l2': 6.2241955723355, 'lambda_l1': 0.03955312698951774, 'top_rate': 0.3900241495765696, 'other_rate': 0.26861025208850203, 'num_iterations': 515}
Best lags: [1, 2, 3, 4, 5]


### Selecting the best ETS (Error, Trend, Seasonality) model usig `optuna_tune` or `hyperopt_tune`

In [5]:
from peshbeen.models import ets

ets_param_space = {
    "smoothing_level":     lambda t: t.suggest_float("smoothing_level", 0.001, 0.99),
    "trend":              lambda t: t.suggest_categorical(
                             "trend", [
                                 "add",
                                 "mul",
                                 None
                             ]),
    "seasonal":           lambda t: t.suggest_categorical(
                             "seasonal", [
                                 "add",
                                 "mul",
                                 None
                             ]),
    "smoothing_trend":    lambda t: t.suggest_float("smoothing_trend", 0.001, 0.99),
    "smoothing_seasonal": lambda t: t.suggest_float("smoothing_seasonal", 0.001, 0.99),
    "smoothing_level":     lambda t: t.suggest_float("smoothing_level", 0.001, 0.99),
    "seasonal_periods":              lambda t: 7,   # fixed, not sampled
        "box_cox":           lambda t: t.suggest_float("box_cox", 0.0, 4),
        "box_cox_biasadj":   lambda t: t.suggest_categorical("box_cox_biasadj", [True, False])
}

ets_model = ets(target_col='admissions')
best_params, _, other_ = optuna_tune(
    model=ets_model,
    df=train,
    cv_split=4,
    step_size=1,
    test_size=30,
    eval_metric=RMSE,
    eval_num=100,
    param_space=ets_param_space, verbose=False
)
print("Best params:", best_params)
print("Other info:", other_)

Best params: {'smoothing_level': 0.43667429119408724, 'trend': None, 'seasonal': None, 'smoothing_trend': 0.8445014317915368, 'smoothing_seasonal': 0.05004223659893368}
Other info: {'box_cox': 2.760234201526317, 'box_cox_biasadj': True}


In [6]:
# now forecast wit the best parameters
best_params.update(other_)
ets_model = ets(target_col='admissions', **best_params)
ets_model.fit(train)
ets_forecasts = ets_model.forecast(H=30)

In [7]:
# get cross validation results with the best parameters
cv_df = ets_model.cross_validate(
    df=train,
    cv_split=5,
        step_size=7,
        test_size=30,
        metrics=[RMSE, MAE])
cv_df.head()

,cutoff,index,split,y_true,y_pred
0,2022-12-07,2022-12-07,fold_1,8933,8931.422872
1,2022-12-07,2022-12-08,fold_1,9013,8931.422872
2,2022-12-07,2022-12-09,fold_1,9000,8931.422872
3,2022-12-07,2022-12-10,fold_1,8821,8931.422872
4,2022-12-07,2022-12-11,fold_1,8835,8931.422872


### Selecting the best orders for an ARIMA model using `optuna_tune` or `hyperopt_tune`

In [8]:
from peshbeen.models import arima
from itertools import product
# Define the hyperparameter search space for ARIMA
p_values = [0, 1, 2, 3]
d_values = [1]
q_values = [0, 1, 2, 3]

# create the list of orders using the product of p, d and q values
orders = list(product(p_values, d_values, q_values))

# Define the hyperparameter search space for seasonal ARIMA
P_values = [0, 1, 2, 3]
D_values = [0, 1]
Q_values = [0, 1, 2, 3]

# create the list of seasonal orders using the product of P, D and Q values
seasonal_orders = list(product(P_values, D_values, Q_values))

# let's define the hyperparameter space for arima using hyperopt
arima_param_space = {
    "order": hp.choice("order", orders),
    "seasonal_order": hp.choice("seasonal_order", seasonal_orders),
    "seasonal_length": 7,
    "box_cox": hp.uniform("box_cox", 0.0, 4),
    "box_cox_biasadj": hp.choice("box_cox_biasadj", [True, False])
}

arima_model = arima(target_col='admissions')
best_params, _, other_ = hyperopt_tune(
    model=arima_model,
    df=train,
    cv_split=10,
    step_size=10,
    test_size=30,
    eval_metric=RMSE,
    eval_num=5,
    param_space=arima_param_space
)

100%|██████████| 5/5 [00:07<00:00,  1.54s/trial, best loss: 145.7250197114811]


### hyperopt_tune example for multivariate forecasting

In [9]:
from peshbeen.datasets import load_admission_calls
from peshbeen.models import ml_mv_forecaster
from peshbeen.metrics import RMSE
from peshbeen.model_selection import mv_hyperopt_tune
from lightgbm import LGBMRegressor
load_admission_calls["day_of_week"] = load_admission_calls.index.dayofweek
load_admission_calls["month"] = load_admission_calls.index.month
train = load_admission_calls[:-30]
test = load_admission_calls[-30:]

cat_variables = ["day_of_week", "month"]
mv_model = ml_mv_forecaster(model=LGBMRegressor(verbose=-1),
              target_cols=['admissions', "calls"], lags = {"admissions": 7, "calls": 7},
                cat_variables=cat_variables,
                difference={"admissions": 1, "calls": 1})
lgb_param_space={'learning_rate': hp.uniform('learning_rate', 0.001, 0.6),
            'num_leaves': scope.int(hp.quniform('num_leaves', 10, 200, 1)),
           'max_depth':scope.int(hp.quniform('max_depth', 2, 18, 1)),
            'bagging_fraction': hp.uniform('bagging_fraction', 0.5, 1),
            'feature_fraction': hp.uniform('feature_fraction', 0.5, 1),
           'min_data_in_leaf': scope.int(hp.quniform ('min_data_in_leaf', 5, 100, 1)), 
            'lambda_l2' : hp.uniform('lambda_l2', 0,10),
           'lambda_l1' : hp.uniform('lambda_l1', 0, 10),
            'other_rate' : hp.quniform('other_rate', 0.05, 0.3, 0.0001),
           'num_iterations': scope.int(hp.quniform("num_iterations", 30, 700, 1)),
           'top_k': scope.int(hp.quniform('top_k', 8, 30, 1)),
                "seed":0, 'lags': hp.choice("lags", [
                                 [1,2,3,4,5],
                                 [1,4,7],
                                 [1,2,3,4,5,6,7],
                                 [1,2,3,4,5,6,7,14]])}
best_params, best_lags = mv_hyperopt_tune(model=mv_model, df=train, target_col= "admissions", cv_split=5, step_size=10,
                                        test_size=30, eval_metric=RMSE, eval_num=4,
                                        param_space=lgb_param_space)
print("Best params:", best_params)
print("Best lags:", best_lags)

100%|██████████| 4/4 [00:07<00:00,  1.93s/trial, best loss: 232.78444693649257]
Best params: {'bagging_fraction': 0.6556625835216798, 'feature_fraction': 0.8726954842592598, 'lambda_l1': 1.6698018914769475, 'lambda_l2': 8.127810960457898, 'learning_rate': 0.2896408049256802, 'max_depth': 17, 'min_data_in_leaf': 18, 'num_iterations': 103, 'num_leaves': 81, 'other_rate': 0.1835, 'seed': 0, 'top_k': 24}
Best lags: {'admissions': [1, 2, 3, 4, 5, 6, 7], 'calls': [1, 2, 3, 4, 5, 6, 7]}


In [10]:
# forecast with the best parameters and best lags
mv_model = ml_mv_forecaster(model=LGBMRegressor(**best_params, verbose=-1),
              target_cols=['admissions', "calls"], lags = best_lags,
                cat_variables=cat_variables,
                difference={"admissions": 1, "calls": 1})
mv_model.fit(train)
mv_forecasts = mv_model.forecast(H=30, exog=test[cat_variables])
mv_forecasts

{'admissions': array([7999.23522708, 8119.77140345, 8115.88771562, 8081.88854924,
        8019.21455733, 7977.4792765 , 7809.94096786, 7895.38821718,
        8031.80470613, 8164.77914105, 8151.15157267, 8095.38862709,
        8027.90474332, 7962.82540976, 8026.99212457, 8162.81049876,
        8130.16382333, 8105.86169817, 8057.9256565 , 7986.18846081,
        7845.15201408, 7865.73387246, 8049.03610717, 8065.12898767,
        8008.71533039, 8028.33310306, 7965.75381092, 7916.76652006,
        7985.32118042, 8128.59909826]),
 'calls': array([1233.38565474, 1293.80100576, 1224.30630467, 1233.79750821,
        1201.50205299, 1272.53397223, 1245.97805409, 1268.14954789,
        1262.21643354, 1194.07953926, 1160.98074865, 1184.11568988,
        1248.19245658, 1262.50863208, 1185.54993692, 1264.99493521,
        1222.78296005, 1226.67223855, 1233.5937844 , 1271.92279809,
        1213.22001578, 1230.43306271, 1284.89399594, 1230.41138678,
        1253.41669914, 1230.17672457, 1346.05003462, 

### optuna_tune example for multivariate forecasting

In [11]:
from peshbeen.model_selection import mv_optuna_tune
lgb_param_space = {
    "learning_rate":     lambda t: t.suggest_float("learning_rate", 0.001, 0.6),
    "num_leaves":        lambda t: t.suggest_int("num_leaves", 10, 200),
    "max_depth":         lambda t: t.suggest_int("max_depth", 2, 18),
    "bagging_fraction":  lambda t: t.suggest_float("bagging_fraction", 0.5, 1.0),
    "feature_fraction":  lambda t: t.suggest_float("feature_fraction", 0.5, 1.0),
    "min_data_in_leaf":  lambda t: t.suggest_int("min_data_in_leaf", 5, 100),
    "lambda_l2":         lambda t: t.suggest_float("lambda_l2", 0.0, 10.0),
    "lambda_l1":         lambda t: t.suggest_float("lambda_l1", 0.0, 10.0),
    "other_rate":        lambda t: t.suggest_float("other_rate", 0.05, 0.3),
    "num_iterations":    lambda t: t.suggest_int("num_iterations", 30, 700),
    "top_k":             lambda t: t.suggest_int("top_k", 8, 30),
    "seed":              lambda t: 0,   # fixed, not sampled
    'lags': lambda t: t.suggest_categorical(
                             "lags", [1,2,3,4,5,
                                [1,2,3,4,5],
                                 [1,4,7],
                                 [1,2,3,4,5,6,7],
                                 [1,2,3,4,5,6,7,14]
                             ]),    
}

mv_model = ml_mv_forecaster(model=LGBMRegressor(verbose=-1),
              target_cols=['admissions', "calls"], lags = {"admissions": 7, "calls": 7},
                cat_variables=cat_variables,
                difference={"admissions": 1, "calls": 1})

best_params, best_lags = mv_optuna_tune(model=mv_model, df=train, target_col= "admissions", cv_split=5, step_size=10,
                                        test_size=30, eval_metric=RMSE, eval_num=4,
                                        param_space=lgb_param_space)

In [12]:
# forecast with the best parameters and best lags from optuna
mv_model = ml_mv_forecaster(model=LGBMRegressor(**best_params, verbose=-1),
              target_cols=['admissions', "calls"], lags = best_lags,
                cat_variables=cat_variables,
                difference={"admissions": 1, "calls": 1})
mv_model.fit(train)
mv_forecasts = mv_model.forecast(H=30, exog=test[cat_variables])
mv_forecasts

{'admissions': array([8051.69321801, 8179.73630203, 8183.18433374, 8149.44991288,
        8300.74756865, 8301.12890477, 8289.9177523 , 8377.78766455,
        8510.76109155, 8495.69510771, 8472.59269009, 8523.04562078,
        8514.38733272, 8535.41653756, 8536.80669185, 8598.6916117 ,
        8688.2680326 , 8732.71652545, 8684.36537779, 8619.87365351,
        8548.28892235, 8582.77246741, 8698.42785463, 8696.26845552,
        8672.59103695, 8783.73480458, 8800.62840925, 8805.84158333,
        8789.10827252, 8859.39025156]),
 'calls': array([1226.01052974, 1296.05030274, 1177.24437704, 1273.49760717,
        1264.90577863, 1293.9892915 , 1282.43951552, 1228.38470348,
        1318.5156583 , 1272.27961981, 1347.09309125, 1280.15114377,
        1373.24384791, 1394.84247639, 1360.05394919, 1420.41294916,
        1371.11863888, 1434.66485762, 1428.63188725, 1450.74251051,
        1381.16812641, 1401.30256706, 1453.1713169 , 1325.0254073 ,
        1380.73414775, 1361.04903082, 1382.29404748, 